In [2]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/customer-transactions-data/transactions.xlsx
/kaggle/input/customer-transactions-data/main.xlsx


Load Excel Files

In [3]:
members_df = pd.read_excel("/kaggle/input/customer-transactions-data/main.xlsx")
txn_df = pd.read_excel("/kaggle/input/customer-transactions-data/transactions.xlsx")

print(members_df.shape, txn_df.shape)


(1048575, 18) (1048575, 24)


Data Cleaning  

In [4]:
# Drop Excel index column if present
members_df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)
txn_df.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

# Remove rows without MEMBER_ID
members_df = members_df[members_df["MEMBER_ID"].notna()]
txn_df = txn_df[txn_df["MEMBER_ID"].notna()]

Convert date column

In [6]:
txn_df["TXN_DT"] = pd.to_datetime(txn_df["TXN_DT"], errors="coerce")

Feature engineering

In [7]:
agg_df = txn_df.groupby("MEMBER_ID").agg(
    total_txns=("TXN_NUM", "count"),
    total_amount=("AMOUNT", "sum"),
    avg_amount=("AMOUNT", "mean"),
    total_points=("POINTS", "sum"),
    active_days=("TXN_DT", lambda x: x.dt.date.nunique())
).reset_index()

agg_df["txns_per_day"] = agg_df["total_txns"] / agg_df["active_days"]

Merge member + transaction features

In [10]:
numeric_cols = final_df.select_dtypes(include=["int64", "float64"]).columns
final_df[numeric_cols] = final_df[numeric_cols].fillna(0)

Select features for ML

In [11]:
feature_cols = [
    "total_txns",
    "total_amount",
    "avg_amount",
    "total_points",
    "active_days",
    "txns_per_day"
]

X = final_df[feature_cols]

Scale features

In [13]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Train Isolation Forest

In [14]:
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)

final_df["anomaly_flag"] = iso_forest.fit_predict(X_scaled)
final_df["anomaly_score"] = iso_forest.decision_function(X_scaled)

Label fraud

In [15]:
final_df["fraud_label"] = final_df["anomaly_flag"].map({
    -1: "Suspicious",
     1: "Normal"
})

final_df[["MEMBER_ID", "fraud_label"]].head()

,MEMBER_ID,fraud_label
0,1-2ZHB77QW,Normal
1,1-72OKPHKT,Normal
2,1-33DXGTHO,Normal
3,1-275KQO2Z,Normal
4,1-9OX0GRXC,Normal


Save output

In [19]:
final_df.to_csv("output.csv", index=False)